# Simulating the UCJ ansatz with fermionic backpropagation

This guide demonstrates how to simulate a one-layer [UCJ ansatz](../explanations/lucj.ipynb) with fermionic backpropagation. 

In [14]:
import warnings
from collections import defaultdict

import pyscf
import pyscf.cc

import numpy as np

import scipy

import ffsim
from ffsim.variational.ucj_energy import ucj_energy, optimize_ucj_energy

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"


# UCJ circuit for a closed-shell molecule
We'll construct the ansatz for a nitrogen molecule in the 6-31g basis set. Since it's a closed-shell system, use the spin-balanced UCJ ansatz. We will restrict the pair connectivity, however fermionic backpropagation in general will work for any connectivity.

In [ ]:
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="6-31g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular data and Hamiltonian
scf = pyscf.scf.RHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf, active_space=active_space)
norb, nelec = mol_data.norb, mol_data.nelec
mol_hamiltonian = mol_data.hamiltonian
print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()

n_reps = 1

# Define interactions
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = (
    None
)

# Use the backend implementable pairs_ab to construct the ucj_op
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    ccsd.t2,
    t1=ccsd.t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization.
    # Additionally, you may want to set the multi_stage_start or multi_stage_step
    # arguments (or both) to obtain a better result at increased computational cost.
    # See the API documentation for details.
    optimize=True,
    options=dict(maxiter=100),
)

Now, let's simulate the ansatz using the backpropagation method. 

In [4]:
energy = ucj_energy(ucj_op, mol_hamiltonian, nelec, (pairs_aa, pairs_ab))
print(f"HF energy: {scf.e_tot:.6f}")
print(f"LUCJ Energy: {energy:.6f}")
print(f"CCSD energy: {ccsd.e_tot:.6f}")

HF energy: -108.835237
LUCJ Energy: -107.680013
CCSD energy: -109.039826


We can also variationally optimize the ansatz parameters to achieve lower ground state energies. This is particularly useful for strongly correlated systems where the CCSD parameters may not be optimal. This optimization is carried out using `optimize_ucj_energy`, which uses JAX for autodifferentiation.

In [17]:
info = defaultdict(list)

def callback(intermediate_result: scipy.optimize.OptimizeResult):
    print(f" {intermediate_result.fun:.6f}")

optimal_ucj, result = optimize_ucj_energy(ucj_op, mol_hamiltonian, nelec, interaction_pairs=(pairs_aa, pairs_ab), return_optimize_result=True, chunk_size = norb**3, callback=callback, options=dict(maxiter=10))

 -108.556534
 -108.746764
 -108.807412
 -108.834293
 -108.840762
 -108.843196
 -108.844902
 -108.846769
 -108.848078
 -108.850054
